# Train baseline and inspect holdout error


In [ ]:
import pandas as pd
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
cvs = pd.read_csv("../data/cvs.csv", encoding="utf-8-sig")
scores = pd.read_csv("../data/scores.csv", sep=";", encoding="utf-8-sig")
scores["candidate_id"] = scores["fichier"].str.removesuffix(".pdf").str.lower()
data = cvs.merge(scores[["candidate_id", "total_sur20"]], on="candidate_id")
model = make_pipeline(TfidfVectorizer(max_features=20000, ngram_range=(1,2)), Ridge(alpha=1.0))
pred = cross_val_predict(model, data.cv_text.fillna(""), data.total_sur20, cv=LeaveOneOut())
mae = abs(pred - data.total_sur20).mean()
print(f"Leave-one-out MAE: {mae:.2f} points over {len(data)} CVs; interpret cautiously.")

# Fit final model


In [ ]:
model.fit(data.cv_text.fillna(""), data.total_sur20)
from pathlib import Path
import joblib
joblib.dump(model, "../models/scorer.pkl")